# POLITE — 2026-07-30 observation

Science-first dual-beam commissioning: matched-beam focus, beam-level S/N, an unpolarized standard, and a polarized-standard repeat at two field-rotator angles.

Configuration: V filter; Mode 5, gain 56, offset 20; 1×1. The server remains full-frame by default. This notebook uses one full-frame twilight probe, then a validated session ROI; the mount is never connected, enabled, homed, slewed, or tracked.

## Operating rules

- Choose and write the two catalog-verified standard names below before opening. Do not select standards from memory at the telescope.
- HWP, focuser, and field-rotator moves use the approved palette helpers and their read-backs are logged. The PWI4 mount is never connected, enabled, homed, slewed, tracked, or parked.
- Stop if either beam clips, either beam approaches the edge margin, the timestamp is implausible, the EFW identity gate fails, or the temperature is unstable.
- This notebook captures one supervised frame at a time; do **not** run `execute_night.py` simultaneously. A saturated HWP frame invalidates its whole four-angle cycle.

## 0 · Shared preamble

This is the template preamble adapted for the extra `observation_notebooks/` directory. Run these two cells first.

In [ ]:
# --- POLITE path bootstrap ---
import os, sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root:
    _root = _root.parent
if not (_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the POLITE repository root')
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print('POLITE root:', _root)


In [ ]:
# --- Shared import set ---
%matplotlib inline
from pathlib import Path
import csv
import re

import numpy as np
import matplotlib.pyplot as plt

from obs_utils import live
from obs_utils import interactive as obs
from obs_utils import user_config as uc
from obs_utils.roi import CameraROI, apply_roi, camera_effective_area

plt.rcParams['figure.dpi'] = 110


## 1 · Tonight's card

The standards, observing mode, and HWP order are fixed below. Fill only measured exposure times and focus positions. Retain a science exposure only if both bias-subtracted beam peaks are below half of the observed digital ceiling and any measured linearity limit.

In [ ]:
NIGHT = '20260730'
SESSION_DIR = Path('FITSDATA') / NIGHT
FILTER_NAME = 'Photometric V'
READOUT_MODE, GAIN, OFFSET = 5, 56, 20   # project default: Mode 5 / gain 56 / offset 20
SETPOINT_C = -15.0
BINNING = 1
HWP_ANGLES = (0.0, 45.0, 22.5, 67.5)  # q beam-swap pair, then u pair

UNPOL_TARGET = 'HD 154892'
POL_TARGET = 'HD 154445'
FOCUS_EXP_S = 0.01       # begin here; increase only if the stellar probes are too faint
SCIENCE_EXP_S = None     # set after the focused stellar probe
TWILIGHT_FLATS_TAKEN = False  # sky was dark before a valid flat sequence could start
FOCUS_POSITIONS = None      # short nominal-centred sweep, entered after first full-frame probe
FOCUS_SELECTED = None       # human-selected position from the focus curve

SESSION_DIR.mkdir(parents=True, exist_ok=True)
print('session:', SESSION_DIR)


## 2 · Bring up camera, EFW, HWP, focuser, and field rotator

**COMMANDS HARDWARE.** The PWI4 client is used only for the focuser and field rotator; no mount operation is issued. On the observatory PC, start the Alpaca services if needed; then connect components independently.

In [ ]:
# Observatory Windows PC only; skip on the lab Mac.
from obs_utils.alpaca_servers import start_observatory_alpaca_servers
start_observatory_alpaca_servers(
    ascom_endpoint=uc.ALPACA_CONFIG.host,
    qhy_endpoint=uc.ALPACA_CONFIG.camera_host,
)


In [ ]:
# COMMANDS CONNECTIONS ONLY — PWI4 is used only for its auxiliary axes.
s = obs.connect_camera()
s = obs.connect_filter_wheel()
s = obs.connect_hwp()           # HWP path, not the PWI4 field rotator
s = obs.connect_focuser()
s = obs.connect_field_rotator()
s.status()


## 3 · Fail-closed configuration gates

**COMMANDS HARDWARE.** Verify the installed EFW order, set the exact observing configuration, and wait for temperature stability. The opaque Dark slot is mandatory for bias and dark frames because the camera has no shutter.

In [ ]:
from obs_utils.night_safety import verify_filter_wheel, cooler_gate, HWP_DEFAULT_TOL_DEG

verify_filter_wheel(s.imaging)
assert s.camera is not None
s.camera.BinX = BINNING
s.camera.BinY = BINNING
s.camera.Gain = GAIN
s.camera.Offset = OFFSET
s.camera.ReadoutMode = READOUT_MODE
readback = {'BinX': s.camera.BinX, 'BinY': s.camera.BinY, 'Gain': s.camera.Gain,
            'Offset': s.camera.Offset, 'ReadoutMode': s.camera.ReadoutMode}
print('read-back:', readback)
assert readback == {'BinX': 1, 'BinY': 1, 'Gain': 56, 'Offset': 20, 'ReadoutMode': 5}
EFFECTIVE_AREA = camera_effective_area(s.camera)
assert (EFFECTIVE_AREA.startx, EFFECTIVE_AREA.starty, EFFECTIVE_AREA.numx, EFFECTIVE_AREA.numy) == (24, 0, 6252, 4176), EFFECTIVE_AREA
FULL_FRAME_ROI = CameraROI(startx=24, starty=0, numx=6252, numy=4176, binx=1, biny=1)
PROVISIONAL_ROI = CameraROI(startx=524, starty=0, numx=5252, numy=4176, binx=1, biny=1)
FULL_FRAME_ROI.validate(s.camera.CameraXSize, s.camera.CameraYSize, effective_area=EFFECTIVE_AREA)
PROVISIONAL_ROI.validate(s.camera.CameraXSize, s.camera.CameraYSize, effective_area=EFFECTIVE_AREA)
ACTIVE_ROI = apply_roi(s.camera, FULL_FRAME_ROI, effective_area=EFFECTIVE_AREA)
print('QHY effective area:', EFFECTIVE_AREA)
print('active ROI:', ACTIVE_ROI)
s.camera.SetCCDTemperature = SETPOINT_C
achieved_temp = cooler_gate(s.camera, SETPOINT_C, tol_c=0.5, stable_s=30.0, timeout_s=900.0, poll_s=5.0, assume_yes=False, verbose=True)
print(f'cooler accepted at {achieved_temp:+.2f} C')


In [ ]:
# COMMANDS HWP MOTION: prove the stage responds before science.
for requested in HWP_ANGLES:
    achieved = s.hwp(requested)
    assert abs(achieved - requested) <= HWP_DEFAULT_TOL_DEG, (requested, achieved)
    print(f'HWP {requested:5.1f} -> {achieved:7.3f} deg')


## 4 · Capture helpers and plain-CSV ledger

The raw FITS are authoritative. `capture_manifest.csv` is a small operator ledger, not a database. `aperture_measurements.csv` is deliberately long-form so the reduction can retain every beam and aperture radius rather than only a final average.

In [ ]:
from alpyca_tools.fits_writer import DetectorCards, FitsHeaderConfig, PolarimetryCards

def _safe(text):
    return re.sub(r'[^A-Za-z0-9]+', '', str(text))

def _next_sequence():
    seqs = []
    for path in SESSION_DIR.glob('*.fits'):
        match = re.match(r'(\d{8})_', path.name)
        if match:
            seqs.append(int(match.group(1)))
    return max(seqs, default=0) + 1

SEQUENCE = _next_sequence()
MANIFEST = SESSION_DIR / 'capture_manifest.csv'
APERTURE_LEDGER = SESSION_DIR / 'aperture_measurements.csv'

if not MANIFEST.exists():
    with MANIFEST.open('w', newline='') as f:
        csv.writer(f).writerow(['filename', 'block', 'target', 'imagetyp', 'exptime_s',
                               'hwp_requested_deg', 'hwp_achieved_deg', 'instrot_deg',
                               'cycle', 'focus_label', 'startx', 'starty', 'numx', 'numy'])
if not APERTURE_LEDGER.exists():
    with APERTURE_LEDGER.open('w', newline='') as f:
        csv.writer(f).writerow(['filename', 'beam', 'aperture_radius_px', 'annulus_rin_px',
                               'annulus_rout_px', 'aperture_sum_adu', 'sky_adu_per_px',
                               'net_source_adu', 'fwhm_px', 'ellipticity', 'peak_adu', 'flag'])

def set_active_roi(roi, *, require_effective_area=True):
    global ACTIVE_ROI
    ACTIVE_ROI = apply_roi(s.camera, roi, effective_area=EFFECTIVE_AREA if require_effective_area else None)
    print('active ROI:', ACTIVE_ROI)
    return ACTIVE_ROI

def capture_one(*, frame_type, block, exposure_s, target=None, hwp_requested=None,
                hwp_achieved=None, instrot_deg=None, cycle=None, focus_label=None):
    """Capture one provenance-rich frame; never commands the mount."""
    global SEQUENCE
    is_dark = frame_type in {'BIAS', 'DARK'}
    s.filter('Dark' if is_dark else FILTER_NAME)
    name_target = _safe(target) + '_' if frame_type == 'LIGHT' and target else ''
    filename_label = {'BIAS': 'Bias', 'DARK': 'Dark', 'FLAT': 'FlatField', 'LIGHT': 'Light'}[frame_type]
    filename = f'{SEQUENCE:08d}_{name_target}{filename_label}_{_safe(FILTER_NAME)}_{exposure_s:.3f}secs'
    if frame_type == 'FLAT':
        filename += '_1x1'
    filename += '.fits'
    pol = PolarimetryCards(hwp_angle_deg=hwp_achieved, instrument_rotator_deg=instrot_deg,
                           pol_seq_id=block if frame_type == 'LIGHT' else None,
                           pol_seq_index=cycle, hwp_uncert_deg=0.012)
    header = FitsHeaderConfig(
        imagetyp='FLAT' if frame_type == 'FLAT' else frame_type,
        object_name=target if frame_type == 'LIGHT' else None, instrument='QHY268M',
        filter_name='Dark' if is_dark else FILTER_NAME, binx=1, biny=1,
        detector=DetectorCards(gain_setting=GAIN, offset_setting=OFFSET, readout_mode=READOUT_MODE,
                                readout_mode_name='Mode 5', cooler_setpoint_c=SETPOINT_C),
        polarimetry=pol,
        extra_cards={'BLOCK': block, 'HWPREQ': hwp_requested, 'FOCUSPOS': str(focus_label or '')},
    )
    path = Path(s.expose(exposure_s, out_path=SESSION_DIR / filename, dark=is_dark, header=header,
                          gain=GAIN, offset=OFFSET, readout_mode=READOUT_MODE,
                          binx=ACTIVE_ROI.binx, biny=ACTIVE_ROI.biny, startx=ACTIVE_ROI.startx,
                          starty=ACTIVE_ROI.starty, numx=ACTIVE_ROI.numx, numy=ACTIVE_ROI.numy))
    with MANIFEST.open('a', newline='') as f:
        csv.writer(f).writerow([path.name, block, target or '', frame_type, exposure_s, hwp_requested,
                                hwp_achieved, instrot_deg, cycle, focus_label or '', ACTIVE_ROI.startx,
                                ACTIVE_ROI.starty, ACTIVE_ROI.numx, ACTIVE_ROI.numy])
    SEQUENCE += 1
    print(path.name)
    return path


## 5 · Twilight-flat decision

**SKIP TONIGHT.** Sky was already dark at 20:55 before a valid twilight sequence could start. Do not take sky flats, a PTC pair, or polarimetric flats now; no valid dark-sky substitute exists. Reduce this point-source data by double ratio. Missing flats never delay the standards.

In [ ]:
assert not TWILIGHT_FLATS_TAKEN
print('Twilight flats/PTC: SKIPPED — proceed directly to the standards.')


## 6 · HD 154445 acquisition, ROI refinement, focus, and exposure selection

Acquire HD 154445 manually. The first stellar frame is deliberately full-frame: measure both beam bounding boxes and one inter-frame drift estimate before entering `SCIENCE_ROI`. It must contain both beams plus the planned burst/cycle drift margin. Run a short nominal-centred PWI4 focuser sweep; inspect both beams in the saved probe and choose the common focus manually. Do not intentionally defocus.

In [ ]:
# MANUAL ACTION: acquire and center both HD 154445 beams at nominal focus, then run this cell.
assert FOCUS_EXP_S is not None and FOCUS_EXP_S > 0
set_active_roi(FULL_FRAME_ROI)
achieved = s.hwp(0.0)
acquisition_path = capture_one(frame_type='LIGHT', block='polstd_fullframe_acquisition', exposure_s=FOCUS_EXP_S, target=POL_TARGET,
                               hwp_requested=0.0, hwp_achieved=achieved, focus_label='nominal')
live.frame_report(acquisition_path)


### ROI checkpoint

Inspect the full-frame acquisition. After confirming both beam boxes and the drift margin fit inside the 500-column crop, run the next cell.

In [ ]:
SCIENCE_ROI = PROVISIONAL_ROI
set_active_roi(SCIENCE_ROI)


### Focus positions

Enter a short nominal-centred set of safe PWI4 positions in the next cell, then run the sweep cell.

In [ ]:
FOCUS_POSITIONS = ()  # replace with at least three safe positions, e.g. (12340, 12360, 12380)
assert len(FOCUS_POSITIONS) >= 3


In [ ]:
achieved = s.hwp(0.0)
sweep = s.focus_sweep(FOCUS_POSITIONS, FOCUS_EXP_S, gain=GAIN, offset=OFFSET, readout_mode=READOUT_MODE,
                      binx=ACTIVE_ROI.binx, biny=ACTIVE_ROI.biny, startx=ACTIVE_ROI.startx, starty=ACTIVE_ROI.starty,
                      numx=ACTIVE_ROI.numx, numy=ACTIVE_ROI.numy)
live.focus_curve(sweep)


### Select focus and science exposure

Inspect both beams and edit `FOCUS_SELECTED` below. After its focused probe, edit `SCIENCE_EXP_S` in the following cell to keep both beam peaks below half the observed ceiling.

In [ ]:
FOCUS_SELECTED = None  # replace with the manually chosen common focus position
assert FOCUS_SELECTED is not None
s.focus(FOCUS_SELECTED)
focused_path = capture_one(frame_type='LIGHT', block='focused_probe', exposure_s=FOCUS_EXP_S, target=POL_TARGET,
                           hwp_requested=0.0, hwp_achieved=achieved, focus_label=str(FOCUS_SELECTED))
live.frame_report(focused_path)


In [ ]:
SCIENCE_EXP_S = None  # replace after inspecting focused_path
assert SCIENCE_EXP_S is not None and SCIENCE_EXP_S > 0


## 7 · Core S/N and polarimetry

The 20-frame HD 154892 burst is physically fixed: do not touch the tube during it. First use the measured cadence/drift to confirm that every aperture remains in `SCIENCE_ROI`; otherwise mark this S/N test infeasible. A four-angle cycle is contiguous—never recenter inside it. Recenter only between cycles, record it in the manifest, and restart a complete cycle after clipping or an edge violation.

In [ ]:
# MANUAL ACTION: center HD 154892 and confirm the drift permits a fixed 20-frame burst.
assert SCIENCE_EXP_S is not None and SCIENCE_EXP_S > 0
achieved = s.hwp(0.0)
for _ in range(20):
    capture_one(frame_type='LIGHT', block='unpol_snr20', exposure_s=SCIENCE_EXP_S, target=UNPOL_TARGET,
                hwp_requested=0.0, hwp_achieved=achieved, instrot_deg=0.0, cycle=0, focus_label='selected')


In [ ]:
def capture_beam_swap_cycle(target, block, cycle, instrot_deg):
    for angle in HWP_ANGLES:
        achieved = s.hwp(angle)
        capture_one(frame_type='LIGHT', block=block, exposure_s=SCIENCE_EXP_S, target=target,
                    hwp_requested=angle, hwp_achieved=achieved, instrot_deg=instrot_deg,
                    cycle=cycle, focus_label='selected')



### Unpolarized beam-swap cycles

Manually recenter HD 154892 before each cell. Each cell captures one complete, uninterrupted four-angle cycle.

In [ ]:
capture_beam_swap_cycle(UNPOL_TARGET, 'unpol_beamswap_rot0', 1, 0.0)


In [ ]:
capture_beam_swap_cycle(UNPOL_TARGET, 'unpol_beamswap_rot0', 2, 0.0)


In [ ]:
capture_beam_swap_cycle(UNPOL_TARGET, 'unpol_beamswap_rot0', 3, 0.0)


In [ ]:
capture_beam_swap_cycle(UNPOL_TARGET, 'unpol_beamswap_rot0', 4, 0.0)


In [ ]:
# MOTION — field rotator only; this never commands the mount.
POL_INSTROT_0 = s.field_rotator_goto_field(0.0)
print('field-rotator read-back:', s.field_rotator_status())


### Polarized standard at 0° field angle

Manually reacquire and center HD 154445, then run one complete cycle cell at a time. Recenter only between cells.

In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot0', 1, POL_INSTROT_0)


In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot0', 2, POL_INSTROT_0)


In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot0', 3, POL_INSTROT_0)


In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot0', 4, POL_INSTROT_0)


In [ ]:
# MOTION — field rotator only; this never commands the mount.
POL_INSTROT_45 = s.field_rotator_goto_field(45.0)
print('field-rotator read-back:', s.field_rotator_status())


### Polarized standard at 45° field angle

Manually reacquire and center HD 154445, then run one complete cycle cell at a time. Recenter only between cells.

In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot45', 1, POL_INSTROT_45)


In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot45', 2, POL_INSTROT_45)


In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot45', 3, POL_INSTROT_45)


In [ ]:
capture_beam_swap_cycle(POL_TARGET, 'polstd_rot45', 4, POL_INSTROT_45)


## 8 · Matching bias and darks

After the stellar blocks, take 20 biases and ten darks at the actual science exposure. All must use the active science ROI and opaque Dark slot. The 5/60/150 s ×3 ladder is optional and runs only after these mandatory frames.

In [ ]:
for _ in range(20):
    capture_one(frame_type='BIAS', block='bias20', exposure_s=0.0)

assert SCIENCE_EXP_S is not None and SCIENCE_EXP_S > 0
for _ in range(10):
    capture_one(frame_type='DARK', block='dark_science', exposure_s=SCIENCE_EXP_S)
# Optional only after all mandatory science and calibration frames:
# for exposure_s in (5.0, 60.0, 150.0):
#     for _ in range(3):
#         capture_one(frame_type='DARK', block=f'dark_optional_{exposure_s:g}s', exposure_s=exposure_s)


## 9 · Live checks and handoff

At the telescope, use these only to detect clipping, temperature drift, or missing HWP angles. The final aperture photometry and S/N comparison belong in the dated reduction notebook. There, write one `aperture_measurements.csv` row per FITS frame, beam, and tested aperture radius.

For each beam, report theoretical S/N including source, sky/dark, read noise, and annular-background-estimate variance; and empirical S/N = mean(net source counts) / sample standard deviation. Reduce each four-angle set with `poltools.modulation.double_ratio`; derive q/u uncertainty directly from complete-cycle scatter. Do not turn the diagnostic HWP flats into a science master flat.

In [ ]:
live.session_table(SESSION_DIR)
all_stats = [live.frame_stats(p) for p in sorted(SESSION_DIR.glob('*.fits'))]
live.temperature_trend(all_stats); plt.show()
live.level_trend(all_stats); plt.show()
live.hwp_coverage(SESSION_DIR)


In [ ]:
from obs_utils.night_safety import verify_filter_wheel
verify_filter_wheel(s.imaging)
print('Raw FITS:', SESSION_DIR)
print('Capture ledger:', MANIFEST)
print('Aperture-ledger schema:', APERTURE_LEDGER)
print('Copy the complete session directory to the external drive before leaving.')


## Shutdown

Release only camera, wheel, and HWP. The mount is never touched.

In [ ]:
obs.shutdown()
